In [1]:
# Run ALL 3 supervised tasks and save CSVs + PKLs to ./supervised
from pathlib import Path
import time, pandas as pd
from project_package.modeling import (
    train_classification_from_csv,
    train_regression_from_csv,
)

ROOT = Path.cwd()
CSV = ROOT / "ncr_ride_bookings_with_weather_filled_scaled_short.csv"
if not CSV.exists():
    alt = ROOT / "datasets" / CSV.name
    if alt.exists(): CSV = alt
assert CSV.exists(), f"CSV not found: {CSV}"

OUT = ROOT / "supervised"
OUT.mkdir(parents=True, exist_ok=True)

def timed(fn, **kw):
    t0 = time.time()
    res = fn(**kw)
    mins = (time.time() - t0) / 60
    return res, mins

def save_report_dict(report: dict, path: Path):
    pd.DataFrame([report]).to_csv(path, index=False)

# 1) Classification: ride completion vs cancellation
cls_res, t_cls = timed(
    train_classification_from_csv,
    csv_path=str(CSV),
    target_col=None,                         # derives _target_completed_ from "Booking Status"
    id_cols=["Booking ID", "Customer ID"],
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=42,
    tune_row_cap=40000,
)
save_report_dict(cls_res.report, OUT / f"cls_best_{cls_res.best_model_name}_report.csv")
print(f"[classification] model={cls_res.best_model_name} time={t_cls:.2f} min")
print("  preds :", cls_res.preds_csv_path)
print("  split :", cls_res.split_csv_path)
print("  model :", cls_res.model_path)  # <-- PKL location

# 2) Regression: fares (Booking Value)
bv_res, t_bv = timed(
    train_regression_from_csv,
    csv_path=str(CSV),
    target_col="Booking Value_fill_scaled",
    id_cols=["Booking ID", "Customer ID"],
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=42,
    tune_row_cap=40000,
)
save_report_dict(bv_res.report, OUT / f"reg_best_Booking_Value_fill_scaled_{bv_res.best_model_name}_report.csv")
print(f"[reg Booking Value] model={bv_res.best_model_name} time={t_bv:.2f} min")
print("  preds :", bv_res.preds_csv_path)
print("  split :", bv_res.split_csv_path)
print("  model :", bv_res.model_path)  # <-- PKL location

# 3) Regression: ratings (Customer Rating)
cr_res, t_cr = timed(
    train_regression_from_csv,
    csv_path=str(CSV),
    target_col="Customer Rating_fill",
    id_cols=["Booking ID", "Customer ID"],
    artifacts_dir=str(OUT),
    valid_ratio=0.20,
    random_state=42,
    tune_row_cap=40000,
)
save_report_dict(cr_res.report, OUT / f"reg_best_Customer_Rating_fill_{cr_res.best_model_name}_report.csv")
print(f"[reg Customer Rating] model={cr_res.best_model_name} time={t_cr:.2f} min")
print("  preds :", cr_res.preds_csv_path)
print("  split :", cr_res.split_csv_path)
print("  model :", cr_res.model_path)  # <-- PKL location


[classification] model=rf time=35.30 min
  preds : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\cls_best_rf_preds.csv
  split : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\split_assignments_classification.csv
  model : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\best_cls_rf__target_completed_.pkl
[reg Booking Value] model=tree time=27.30 min
  preds : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\reg_best_Booking_Value_fill_scaled_tree_preds.csv
  split : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\split_assignments_regression_Booking_Value_fill_scaled.csv
  model : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\best_reg_tree_Booking Value_fill_scaled.pkl
[reg Customer Rating] model=tree time=27.49 min
  preds : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\reg_best_Customer_Rating_fill_tree_preds.csv
  split : c:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis\supervised\split